In [ ]:
using BayesSoundSource
using Distributions 
using LinearAlgebra

function rand_source(std) 
    x = rand(Normal(0.0, std))
    y = rand(Normal(0.0, std))
    z = rand(truncated(Normal(0.0, std); lower=0.0))
    [x,y,z]
end 


## Set receivers to desired geometry 
receivers = [
    [8.0, 0.0, 2.0],
    [0.0, 8.0, 4.0],
    [-8.0, 0.0, 2.0],
    [0.0, -8.0, 4.0],
    [0.0, 0.0, 0.0]
] 

noise = 0.0003
speed_of_sound = 343.0

loss = mean(1:10_000) do i
    source = rand_source(10.0)
    crlb = crlb_tdoa(receivers, source, speed_of_sound=speed_of_sound, σ=noise)
    loss = √tr(crlb)
end 

In [ ]:

using LinearAlgebra
using GLMakie
using LazyGrids 

max_gdop = 100

x_ax = range(-12,12, 50)
y_ax = range(-12,12, 50)
z_ax = range(0,12, 50)

function f(x,y,z)
    source = [x, y, z]
    if source ∈ eachrow(receivers)
        return max_gdop
    end 

    gdop = gdop_tdoa(receivers, source)
    
    # gdop
    return gdop <= max_gdop ? gdop : max_gdop
end 

X, Y, Z = ndgrid(x_ax, y_ax, z_ax)
grid = f.(X, Y, Z)


fig = Figure()

ax = Axis3(fig[1, 1], title="TDoA GDOP, Expected loss = $(round(mean(grid), digits=2))")
GLMakie.contour!(ax, -12 .. 12, -12 .. 12, 0 .. 12, log.(grid), levels=10, alpha=0.5)
GLMakie.scatter!(ax, receivers)
Colorbar(fig[1, 2], label = "GDOP", limits = extrema(grid))
fig